# Today's Lab: Autoencoder vs Variational Autoencoder on Fashion-MNIST

In this lab, you will compare a plain autoencoder (AE) and a variational autoencoder (VAE) on the same dataset.

**Lab question**
- Does a VAE learn a latent space that is useful not only for reconstruction, but also for generation and interpolation?


## What students will do

- Train a plain autoencoder and a VAE on Fashion-MNIST.
- Compare reconstruction quality, latent structure, and generated samples.
- Explore interpolation to see whether the latent space is smooth and meaningful.

**Instructions**
- Complete the `TODO` parts.
- Run the notebook section by section.
- Answer the reflection questions at the end.


In [ ]:
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid


In [ ]:
def set_seed(seed: int = 42) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
@dataclass
class Config:
    """Configuration values for the lab."""
    batch_size: int = 128
    hidden_dim: int = 256
    latent_dim: int = 2
    lr: float = 1e-3
    epochs: int = 5
    subset_train: int | None = 12000
    subset_test: int | None = 2000


cfg = Config()
cfg


## Load Fashion-MNIST

We use a smaller subset so the lab runs quickly on ordinary machines.


In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)

if cfg.subset_train is not None:
    train_dataset = Subset(train_dataset, range(cfg.subset_train))
if cfg.subset_test is not None:
    test_dataset = Subset(test_dataset, range(cfg.subset_test))

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print('Train size:', len(train_dataset))
print('Test size:', len(test_dataset))


In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, img, label in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(img.squeeze(0), cmap='gray')
    ax.set_title(class_names[int(label)], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Plain autoencoder baseline

This model is fully provided so you can focus on how the VAE differs.


In [ ]:
class AutoEncoder(nn.Module):
    """Simple fully connected autoencoder baseline."""

    def __init__(self, input_dim: int = 28 * 28, hidden_dim: int = 256, latent_dim: int = 2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode an image into latent space."""
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode a latent vector into image space."""
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return reconstruction and latent vector."""
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


def ae_loss(recon_x: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Compute reconstruction loss for the plain autoencoder."""
    return F.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)


## VAE skeleton

Complete the VAE-specific components below.


In [ ]:
class VAE(nn.Module):
    """Variational autoencoder skeleton."""

    def __init__(self, input_dim: int = 28 * 28, hidden_dim: int = 256, latent_dim: int = 2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.encoder_backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )

        # TODO 1:
        # Define two layers that output mu and logvar.
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return posterior parameters mu and logvar."""
        h = self.encoder_backbone(self.flatten(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample z via the reparameterization trick."""
        # TODO 2:
        # Implement z = mu + std * eps.
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + std * eps
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector into image space."""
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return reconstruction, mu, logvar, and sampled z."""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z


def vae_loss(recon_x: torch.Tensor, x: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor):
    """Compute total VAE loss."""
    recon = F.binary_cross_entropy(recon_x, x, reduction='sum') / x.size(0)

    # TODO 3:
    # Add the KL divergence term.
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)

    total = recon + kl
    return total, recon, kl


## Training functions


In [ ]:
def train_ae(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    """Train the plain autoencoder for one epoch."""
    model.train()
    total_loss = 0.0
    for x, _ in loader:
        x = x.to(device)
        optimizer.zero_grad()
        recon, _ = model(x)
        loss = ae_loss(recon, x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)


def eval_ae(model: nn.Module, loader: DataLoader) -> float:
    """Evaluate the plain autoencoder."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            recon, _ = model(x)
            loss = ae_loss(recon, x)
            total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)


def train_vae(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer):
    """Train the VAE for one epoch."""
    model.train()
    total_total, total_recon, total_kl = 0.0, 0.0, 0.0
    for x, _ in loader:
        x = x.to(device)
        optimizer.zero_grad()
        recon, mu, logvar, _ = model(x)
        loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar)
        loss.backward()
        optimizer.step()
        bs = x.size(0)
        total_total += loss.item() * bs
        total_recon += recon_loss.item() * bs
        total_kl += kl_loss.item() * bs
    n = len(loader.dataset)
    return total_total / n, total_recon / n, total_kl / n


def eval_vae(model: nn.Module, loader: DataLoader):
    """Evaluate the VAE."""
    model.eval()
    total_total, total_recon, total_kl = 0.0, 0.0, 0.0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            recon, mu, logvar, _ = model(x)
            loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar)
            bs = x.size(0)
            total_total += loss.item() * bs
            total_recon += recon_loss.item() * bs
            total_kl += kl_loss.item() * bs
    n = len(loader.dataset)
    return total_total / n, total_recon / n, total_kl / n


## Train both models


In [ ]:
ae = AutoEncoder(hidden_dim=cfg.hidden_dim, latent_dim=cfg.latent_dim).to(device)
vae = VAE(hidden_dim=cfg.hidden_dim, latent_dim=cfg.latent_dim).to(device)

ae_opt = torch.optim.Adam(ae.parameters(), lr=cfg.lr)
vae_opt = torch.optim.Adam(vae.parameters(), lr=cfg.lr)


In [ ]:
ae_hist = {'train': [], 'test': []}
vae_hist = {'train_total': [], 'train_recon': [], 'train_kl': [], 'test_total': [], 'test_recon': [], 'test_kl': []}

for epoch in range(1, cfg.epochs + 1):
    ae_train = train_ae(ae, train_loader, ae_opt)
    ae_test = eval_ae(ae, test_loader)
    ae_hist['train'].append(ae_train)
    ae_hist['test'].append(ae_test)

    vae_train_total, vae_train_recon, vae_train_kl = train_vae(vae, train_loader, vae_opt)
    vae_test_total, vae_test_recon, vae_test_kl = eval_vae(vae, test_loader)
    vae_hist['train_total'].append(vae_train_total)
    vae_hist['train_recon'].append(vae_train_recon)
    vae_hist['train_kl'].append(vae_train_kl)
    vae_hist['test_total'].append(vae_test_total)
    vae_hist['test_recon'].append(vae_test_recon)
    vae_hist['test_kl'].append(vae_test_kl)

    print(
        f'Epoch {epoch:02d} | '
        f'AE test {ae_test:.4f} | '
        f'VAE total {vae_test_total:.4f} | '
        f'recon {vae_test_recon:.4f} | '
        f'kl {vae_test_kl:.4f}'
    )


In [ ]:
epochs = range(1, cfg.epochs + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, ae_hist['test'], label='AE test loss')
plt.plot(epochs, vae_hist['test_total'], label='VAE total loss')
plt.plot(epochs, vae_hist['test_recon'], label='VAE recon loss', linestyle='--')
plt.plot(epochs, vae_hist['test_kl'], label='VAE KL loss', linestyle=':')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss curves')
plt.legend()
plt.tight_layout()
plt.show()


## Compare reconstructions


In [ ]:
def show_reconstructions(ae_model: nn.Module, vae_model: nn.Module, loader: DataLoader, n: int = 8) -> None:
    """Show original images and reconstructions from both models."""
    ae_model.eval()
    vae_model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    with torch.no_grad():
        ae_recon, _ = ae_model(x)
        vae_recon, _, _, _ = vae_model(x)

    fig, axes = plt.subplots(3, n, figsize=(1.5 * n, 4.5))
    for i in range(n):
        axes[0, i].imshow(x[i].cpu().squeeze(), cmap='gray')
        axes[1, i].imshow(ae_recon[i].cpu().squeeze(), cmap='gray')
        axes[2, i].imshow(vae_recon[i].cpu().squeeze(), cmap='gray')
        for r in range(3):
            axes[r, i].axis('off')
    axes[0, 0].set_ylabel('Input')
    axes[1, 0].set_ylabel('AE')
    axes[2, 0].set_ylabel('VAE')
    plt.tight_layout()
    plt.show()


show_reconstructions(ae, vae, test_loader)


## Visualize latent space

For the VAE, we visualize the posterior mean `mu`.


In [ ]:
def collect_latents_ae(model: nn.Module, loader: DataLoader):
    """Collect AE latent vectors and labels."""
    model.eval()
    zs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            _, z = model(x)
            zs.append(z.cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(zs), np.concatenate(ys)


def collect_latents_vae(model: nn.Module, loader: DataLoader):
    """Collect VAE posterior means and labels."""
    model.eval()
    zs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            mu, _ = model.encode(x)
            zs.append(mu.cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(zs), np.concatenate(ys)


In [ ]:
ae_z, ae_y = collect_latents_ae(ae, test_loader)
vae_z, vae_y = collect_latents_vae(vae, test_loader)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(ae_z[:, 0], ae_z[:, 1], c=ae_y, s=8, cmap='tab10', alpha=0.7)
axes[0].set_title('AE latent space')
axes[1].scatter(vae_z[:, 0], vae_z[:, 1], c=vae_y, s=8, cmap='tab10', alpha=0.7)
axes[1].set_title('VAE latent space (mu)')
for ax in axes:
    ax.set_xlabel('z1')
    ax.set_ylabel('z2')
plt.tight_layout()
plt.show()


## Sample from the VAE prior


In [ ]:
def sample_from_vae(model: nn.Module, n: int = 16) -> None:
    """Sample new images from a standard normal prior."""
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, cfg.latent_dim, device=device)
        samples = model.decode(z).cpu()
    grid = make_grid(samples, nrow=4, pad_value=1.0)
    plt.figure(figsize=(5, 5))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    plt.axis('off')
    plt.title('Random VAE samples')
    plt.show()


sample_from_vae(vae)


## Latent interpolation

Interpolate between two images in latent space.


In [ ]:
def latent_interpolation(model: nn.Module, x1: torch.Tensor, x2: torch.Tensor, steps: int = 8) -> torch.Tensor:
    """Interpolate between two latent representations and decode the path."""
    model.eval()
    with torch.no_grad():
        mu1, _ = model.encode(x1.to(device))
        mu2, _ = model.encode(x2.to(device))
        alphas = torch.linspace(0.0, 1.0, steps, device=device).view(-1, 1)

        # TODO 4:
        # Inspect this interpolation formula and explain why it works.
        z = (1 - alphas) * mu1 + alphas * mu2

        out = model.decode(z)
    return out.cpu()


In [ ]:
x_batch, y_batch = next(iter(test_loader))
interp = latent_interpolation(vae, x_batch[0:1], x_batch[1:2], steps=10)

fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for ax, img in zip(axes, interp):
    ax.imshow(img.squeeze(0), cmap='gray')
    ax.axis('off')
plt.suptitle('VAE interpolation')
plt.tight_layout()
plt.show()


## Reflection questions

Write short answers below each question.

1. Which model gives sharper reconstructions?
2. Which model produces a more organized latent space?
3. Are random VAE samples plausible?
4. What does the KL term encourage?
5. Why can the VAE support interpolation more naturally than a plain AE?


In [ ]:
# Write your answers here.
